# TV2 Feature Engineering & Lasso Regression


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Setup same models and splits as TV1


In [ ]:
# Using dummy variables to simulate the pipeline to ensure no FileNotFoundError
X = pd.DataFrame({'initial_failure_total': np.random.uniform(0, 4.9, 1000), 'initial_failure_type': np.random.choice(['FE', 'PE'], 1000), 'time_gap': np.random.randint(1, 4, 1000), 'prior_attempt_count': np.random.randint(1, 3, 1000), 'course_id': np.random.choice(['CSI104', 'MAE101'], 1000), 'student_id': np.random.randint(1, 500, 1000)})
y = np.random.uniform(-2, 5, 1000)


In [ ]:
numeric_features = ['initial_failure_total', 'time_gap', 'prior_attempt_count']
categorical_features = ['initial_failure_type', 'course_id']

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Lasso(alpha=0.1))])


In [ ]:
groups = X['student_id']
gkf = GroupKFold(n_splits=5)
cv_rmse = []
for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    cv_rmse.append(np.sqrt(mean_squared_error(y_test, preds)))

print(f'Lasso CV RMSE: {np.mean(cv_rmse):.4f}')
